DATA EXPLORATION

In [ ]:
import pandas as pd

dengue = pd.read_csv('Dengue_diseases_dataset_modified (1).csv')
datadict = pd.read_csv('data_dictionary.csv')

In [ ]:
print(datadict)

In [ ]:
print("Shape:", dengue.shape)
print("Column Names:", dengue.columns.tolist())
print("Column Types:", dengue.dtypes)

In [ ]:
dengue['dengue_label'].value_counts() 
#644 + cases, 345 - cases 

In [ ]:
dengue.isnull().sum()
#Some missing values in WBC count, platelet count, platelet distribution width
#Otherwise no missing

In [ ]:
dengue.isnull().sum() / len(dengue) * 100  
#Less than 2.5% missing for all the missing variables

#Options:
#Remove rows with missing values
#Fill with the mean
#KNN imputation

DATA CLEANING


In [ ]:
#Changing gender column to binary columns
dummies = pd.get_dummies(dengue['gender'], prefix=None).astype(int)
dengue = pd.concat([dengue, dummies], axis=1)
dengue = dengue.drop(columns=['gender'])


In [ ]:
#KNN imputation
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
dengue_imputed = pd.DataFrame(
    imputer.fit_transform(dengue),
    columns=dengue.columns
)

RAW CORRELATIONS

In [ ]:
#First look, without any cleaning
#Does not include sex/childhood status, beacuse that's categorical

corr_matrix = dengue.corr(numeric_only=True)
import seaborn as sns
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')

#Immediatley, WBC count and platelet count seem to be the most correlated with dengue fever. 
#Both negatively correlated - lower WBC count, higher chance of dengue

LOGISTIC REGRESSION

In [ ]:
from sklearn.model_selection import train_test_split

X = dengue_imputed.drop(columns=['dengue_label'])  # features
y = dengue_imputed['dengue_label']                 # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

#rather poor performance

coefficients = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_[0]
})


In [ ]:
#coefficients
print(coefficients.sort_values('coefficient', ascending=False))

#hemoglobin the primary predictor, all else small by comparison

RANDOM FOREST CLASSIFIER

In [ ]:
#Random forest classifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))

#Not much better

In [ ]:
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
})

print(importances.sort_values('importance', ascending=False))

#WBC count, platelet count, and platelet distribution width the most significant predictors

In [ ]:
#Plotting features
import matplotlib.pyplot as plt
importances.sort_values('importance').plot(
    kind='barh', x='feature', y='importance', legend=False
)

plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

XGBOOST
- XGBoost builds decision trees one at a time, each new one attempting to correct the mistakes of the previous tree

In [ ]:
#Running XGBOOST
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

xgb_model = XGBClassifier(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb))
#not all that much better

In [ ]:
#Plotting important features
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
})
print(importances.sort_values('importance', ascending=False))

#WBC count by far the most important feature 

In [ ]:
#Plotting features
import matplotlib.pyplot as plt
importances.sort_values('importance').plot(
    kind='barh', x='feature', y='importance', legend=False
)

plt.title('XGBOOST Feature Importances')
plt.tight_layout()
plt.show()

KNN Classification

In [ ]:
#Scaling the data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Train the model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred_knn = knn_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred_knn))

In [ ]:
#Find optimal K
import matplotlib.pyplot as plt

error_rates = []
for k in range(1, 30):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    error_rates.append(1 - knn.score(X_test_scaled, y_test))

plt.plot(range(1, 30), error_rates, marker='o')
plt.xlabel('K value')
plt.ylabel('Error Rate')
plt.title('KNN Error Rate by K')
plt.show()

#Lets do best k = 5

In [ ]:
#Refit with best K
knn_best = KNeighborsClassifier(n_neighbors=5)  # replace best_k with your chosen value
knn_best.fit(X_train_scaled, y_train)
y_pred_best = knn_best.predict(X_test_scaled)
print(classification_report(y_test, y_pred_best))
#about the same